# RAG 챗봇 — 내 문서에 답하는 AI

**군-학 연계 프로그램 범용 교육 3차시** · 민트상사 사내 규정집(가상 문서)으로 문서 질의응답 챗봇을 만듭니다.

## 사용 규칙 3가지
1. **위에서 아래로**, 셀 왼쪽의 ▶ 버튼을 하나씩 누릅니다. 셀이 끝나면 왼쪽 `[ ]` 안에 숫자가 생깁니다.
2. 바꿔 봐도 되는 줄은 `# <-- 바꿔 보세요` 표시가 있는 줄뿐입니다.
3. 빨간 글씨(에러)가 나오면 그대로 두고 손을 듭니다. 대부분 "셀 1을 안 돌림" 또는 "순서를 건너뜀"입니다.

| 셀 | 하는 일 | 걸리는 시간 |
|---|---|---|
| 1 | 준비(도구 설치) | 약 1분 |
| 2 | 문서 불러오기 | 바로 |
| 3 | 자르기(청킹) | 바로 |
| 4 | 숫자로(임베딩) + 유사도 체험 | 첫 실행 1~2분 |
| 5 | 찾기(검색) | 바로 |
| 6 | 조립(프롬프트 만들기) → claude.ai에 붙여넣기 | 바로 |
| 7 | 답하기(생성 모델, GPU가 있을 때) | 1~2분 |
| 8 | 챗봇(질문 → 답, 되풀이) | 바로 |
| 9 | (선택) 내 문서로 바꾸기 | 바로 |

## 셀 1. 준비 — 필요한 도구를 설치합니다
▶를 누르고 "준비 완료"가 나올 때까지 기다립니다(약 1분).

In [ ]:
# 셀 1. 준비 — 필요한 도구를 설치합니다 (약 1분)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentence-transformers", "pypdf"], check=True)
print("준비 완료")

## 셀 2. 문서 — 오늘의 문서(민트상사 사내 규정집)를 불러옵니다
문서는 이 노트북 안에 들어 있어 따로 올릴 필요가 없습니다. 실제 회사가 아니라 수업용 가상 규정집입니다.

In [ ]:
# 셀 2. 문서 — 민트상사 사내 규정집(가상)을 DOCUMENT 라는 이름으로 담습니다
DOCUMENT = """민트상사 사내 규정집 (2026년 1월 1일 개정판)

제1장 총칙

제1조(목적) 이 규정은 민트상사(이하 "회사") 임직원의 근무, 휴가, 출장, 경비 처리, 재택근무, 정보보안, 복리후생에 관한 기준을 정한다.

제2조(적용 범위) 이 규정은 정규직과 계약직 전 임직원에게 적용한다. 파견 인력과 협력사 인력에게는 제6장(재택근무와 정보보안) 중 정보보안 조항만 적용한다.

제3조(용어) "팀장"은 팀 단위 조직의 장을 말하고, "부서장"은 본부 단위 조직의 장을 말한다. "근무일"은 토요일, 일요일, 공휴일을 제외한 날을 말한다.

제2장 근무

제4조(근무시간) 기본 근무시간은 09시부터 18시까지이며, 점심시간은 12시부터 13시까지 1시간이다. 주 40시간 근무를 기준으로 한다.

제5조(유연근무) 직원은 08시부터 10시 사이에서 출근 시각을 선택할 수 있다. 출근 후 점심시간 1시간을 포함해 9시간을 근무한다. 다음 달의 출근 시각은 매월 말일까지 팀장에게 신청한다.

제6조(초과근무) 초과근무는 팀장의 사전 승인이 있어야 하며, 1일 3시간, 1주 12시간을 넘을 수 없다. 22시 이후의 근무는 부서장 승인이 추가로 필요하다.

제7조(지각과 조퇴) 한 달에 3회 이상 지각하면 팀장 면담을 실시한다. 조퇴는 당일 오전 중에 팀장에게 알린다.

제3장 휴가

제8조(연차휴가) 1. 입사 1년 미만 직원은 1개월 개근할 때마다 1일의 연차가 생긴다. 2. 입사 1년 이상 직원에게는 연 15일의 연차가 주어진다. 3. 근속 3년 이상 직원은 2년마다 1일씩 더해 최대 25일까지 늘어난다. 4. 연차는 사용일 3근무일 전까지 휴가 시스템으로 신청한다. 다만 1일 이하의 연차는 전날 17시까지 신청할 수 있다.

제9조(반차) 반차는 오전(09시부터 13시)과 오후(14시부터 18시)로 나누어 쓴다. 반차 2회는 연차 1일로 계산한다.

제10조(경조휴가) 경조휴가 일수는 다음과 같다. 본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 부모·배우자·자녀 사망 5일, 조부모·형제자매 사망 3일. 경조휴가는 사유 발생일로부터 30일 이내에 사용한다. 증빙 서류는 복귀 후 7일 이내에 제출한다.

제11조(병가) 연 10일까지 유급 병가를 쓸 수 있다. 3일 이상 연속으로 병가를 쓰면 진단서를 제출한다.

제12조(공가) 예비군·민방위 훈련, 공직 선거 투표, 법원 출석은 공가로 처리하며 급여를 감하지 않는다.

제13조(휴가 이월) 사용하지 않은 연차는 다음 해로 이월되지 않으며, 미사용분은 12월 급여에 수당으로 지급한다. 다만 회사가 연차 사용을 촉진했는데도 쓰지 않은 경우에는 수당을 지급하지 않는다.

제4장 출장

제14조(출장 승인) 국내 출장은 팀장, 해외 출장은 부서장의 승인을 출발 3근무일 전까지 받는다.

제15조(국내 출장비) 국내 출장비는 1일 기준 다음 한도 안에서 실비로 정산한다.
- 교통비: 실비. KTX는 일반실, 자차 이용 시 1km당 300원.
- 숙박비: 1박 80,000원 한도.
- 식비: 1일 25,000원 한도.
- 일비(잡비): 1일 20,000원 정액.
숙박이 없는 당일 출장은 식비와 일비만 지급한다.

제16조(해외 출장비) 항공권은 이코노미석을 원칙으로 한다. 비행시간이 8시간 이상이면 부서장 승인을 받아 프리미엄 이코노미석을 이용할 수 있다. 해외 숙박비와 식비 한도는 지역별 별표를 따르며, 별표는 총무팀이 보관한다.

제17조(출장 보고) 출장에서 복귀한 뒤 5근무일 이내에 출장 보고서와 영수증을 제출한다. 기한을 넘기면 다음 달 정산으로 넘어간다.

제5장 경비 처리

제18조(법인카드) 회사 경비는 법인카드로 결제하는 것을 원칙으로 한다. 개인카드로 결제한 경우에는 영수증을 첨부해 경비 청구서를 제출한다.

제19조(사전 승인) 한 건에 100,000원을 초과하는 지출은 사전에 팀장 승인을 받는다. 다만 출장 중의 긴급한 지출은 복귀 후 3근무일 이내의 사후 승인으로 갈음할 수 있다.

제20조(영수증 제출) 영수증은 결제일로부터 7일 이내에 경비 시스템에 등록한다. 결제일로부터 30일이 지난 영수증은 정산하지 않는다.

제21조(접대비) 접대비는 1인당 50,000원 한도이며, 참석자의 이름과 소속을 기록한다. 주류를 포함한 접대는 부서장 승인이 필요하다.

제22조(경비 부인) 개인 용도의 지출, 교통 위반 과태료, 영수증이 없는 지출은 회사가 부담하지 않는다.

제6장 재택근무와 정보보안

제23조(재택근무) 직원은 주 2일까지 재택근무를 할 수 있다. 재택근무는 전날 17시까지 팀장에게 신청하고, 근무 시작과 종료를 사내 메신저로 알린다. 수습 기간 중인 직원은 재택근무를 신청할 수 없다.

제24조(문서 반출) 대외비 이상의 문서는 사외로 반출하거나 개인 이메일, 외부 AI 서비스에 입력할 수 없다. 반출이 꼭 필요하면 정보보안팀의 승인을 받는다.

제25조(비밀번호) 업무 시스템의 비밀번호는 90일마다 변경하며, 10자 이상으로 문자, 숫자, 특수문자를 섞어 만든다.

제26조(분실 신고) 노트북, 휴대폰 등 업무 기기를 분실하면 즉시 정보보안팀(내선 7700)에 신고한다.

제7장 복리후생

제27조(중식 지원) 근무일 1일 10,000원의 중식비를 식대 카드로 지원한다. 재택근무일에도 지급한다.

제28조(자기계발비) 연 500,000원까지 도서 구입비, 강의 수강료, 자격증 응시료를 지원한다. 영수증과 수료증(또는 성적표)을 제출한다. (2026년 1월 1일 개정: 종전 300,000원에서 상향)

제29조(건강검진) 매년 1회 회사가 지정한 기관에서 건강검진을 받는다. 검진일은 공가로 처리한다.

제30조(경조금) 본인 결혼 300,000원, 자녀 출생 200,000원, 부모·배우자·자녀 사망 500,000원을 지급한다. 경조금은 경조휴가 신청과 함께 신청한다.

부칙

1. 이 규정은 2026년 1월 1일부터 시행한다.
2. 개정 이력: 2024년 3월 1일 재택근무(제23조)를 주 1일에서 주 2일로 변경. 2026년 1월 1일 자기계발비(제28조)를 300,000원에서 500,000원으로, 국내 출장 숙박비(제15조)를 70,000원에서 80,000원으로 변경.
3. 이 규정에서 정하지 않은 사항은 취업규칙과 관련 법령을 따른다."""

import re
n_articles = len(re.findall(r"제\d+조\(", DOCUMENT))
print(f"글자 수: {len(DOCUMENT):,}자 / 조항 수: {n_articles}개")
print("--- 앞부분 ---")
print(DOCUMENT[:160])

## 셀 3. 자르기(청킹) — 문서를 조각으로 나눕니다
규정집은 **조항 단위**로 자르는 것이 가장 자연스럽습니다. 4부 실습 7에서 `CHUNK_MODE`를 `"글자수"`로 바꿔 무엇이 달라지는지 봅니다.

In [ ]:
# 셀 3. 자르기(청킹) — 문서를 조각(chunk)으로 나눕니다
CHUNK_MODE = "조항"      # <-- 바꿔 보세요: "조항" 또는 "글자수"
CHUNK_SIZE = 200         # <-- "글자수" 방식일 때 한 조각의 길이(글자)

import re

def split_by_article(text):
    '''"제N조(...)" 마다 한 조각. 조각 앞에 [장 이름]을 붙여 문맥을 남긴다.'''
    titles, chunks, chapter = [], [], ""
    for para in text.split("\n\n"):
        para = para.strip()
        if not para:
            continue
        if re.match(r"제\d+장 ", para):          # 장 제목은 조각이 아니라 꼬리표
            chapter = para
            continue
        m = re.match(r"(제\d+조\([^)]*\))", para)
        if m:
            titles.append(m.group(1))
            chunks.append(f"[{chapter}] {para}")
        elif para.startswith("부칙"):
            titles.append("부칙")
            chunks.append(para)
        # 그 밖의 문단(맨 위 제목 줄 등)은 버린다
    return titles, chunks

def split_by_chars(text, size, overlap=40):
    '''size 글자마다 자르고, 앞 조각 끝 overlap 글자를 겹친다.'''
    text = re.sub(r"\s+", " ", text)
    titles, chunks, start, i = [], [], 0, 1
    while start < len(text):
        chunks.append(text[start:start + size])
        titles.append(f"조각 {i}")
        start += size - overlap
        i += 1
    return titles, chunks

if CHUNK_MODE == "조항":
    chunk_titles, chunks = split_by_article(DOCUMENT)
else:
    chunk_titles, chunks = split_by_chars(DOCUMENT, CHUNK_SIZE)

print(f"조각 수: {len(chunks)}개  (방식: {CHUNK_MODE})")
for i in range(3):
    print(f"[{i+1}] {chunks[i][:70]}...")
if len(chunks) <= 1:
    print('조각이 하나뿐입니다. 조항 형식이 아닌 문서면 CHUNK_MODE = "글자수" 로 바꾸세요.')

## 셀 4. 숫자로(임베딩) — 조각마다 숫자 384개를 만듭니다
첫 실행 때 임베딩 모델(`intfloat/multilingual-e5-small`, 무료·공개)을 내려받습니다(1~2분).
비슷한 뜻의 문장은 비슷한 숫자가 됩니다. 아래에서 문장 3개의 **유사도**(1에 가까울수록 비슷)를 확인합니다.

In [ ]:
# 셀 4. 숫자로(임베딩) — 조각마다 숫자 384개(벡터)를 만듭니다
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("intfloat/multilingual-e5-small")
# e5 모델 규칙: 문서 조각에는 "passage: ", 질문에는 "query: " 를 앞에 붙인다
chunk_vectors = model.encode(["passage: " + c for c in chunks], normalize_embeddings=True)
print("조각 벡터 모양:", chunk_vectors.shape, "= (조각 수, 숫자 개수)")
print("첫 조각의 앞 5개 숫자:", np.round(chunk_vectors[0][:5], 3))

# --- 유사도 체험: 문장 3개 ---
sentences = [
    "국내 출장 숙박비 한도는 1박에 얼마인가요?",
    "호텔 비용은 얼마까지 회사가 내 주나요?",
    "비밀번호는 몇 자 이상이어야 하나요?",
]
vecs = model.encode(["query: " + s for s in sentences], normalize_embeddings=True)
sims = vecs @ vecs.T
print("\n문장끼리 유사도 (1 = 같은 뜻)")
for i, s in enumerate(sentences):
    print(f"  문장{i+1} {np.round(sims[i], 2)}  {s}")

# --- 질문 하나 vs 조각 3개 ---
q = model.encode(["query: " + sentences[0]], normalize_embeddings=True)[0]
print("\n질문:", sentences[0])
for title in ["제15조(국내 출장비)", "제10조(경조휴가)", "제25조(비밀번호)"]:
    if title in chunk_titles:
        i = chunk_titles.index(title)
        print(f"  {title:<14s} 유사도 {float(chunk_vectors[i] @ q):.2f}")

## 셀 5. 찾기(검색) — 질문과 가장 가까운 조각 3개를 고릅니다
`QUESTION` 줄의 따옴표 안 문장을 바꿔 가며 실행합니다. 점수는 상대 비교용입니다.

In [ ]:
# 셀 5. 찾기(검색) — 질문 벡터와 모든 조각 벡터의 유사도를 재서 상위 k개를 고릅니다
def search(question, k=3):
    q = model.encode(["query: " + question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    order = np.argsort(-scores)[:k]
    return [(float(scores[i]), chunk_titles[i], chunks[i]) for i in order]

QUESTION = "국내 출장 숙박비 한도는 1박에 얼마인가요?"   # <-- 바꿔 보세요

print("질문:", QUESTION)
for rank, (score, title, chunk) in enumerate(search(QUESTION), 1):
    print(f"  {rank}위  점수 {score:.2f}  {title}")
    print(f"       {chunk[:48]}...")

## 셀 6. 조립 — 규칙 + 찾은 조각 + 질문을 하나의 프롬프트로 만듭니다
출력된 글 전체를 복사해서 **claude.ai 새 대화**에 붙여 넣으면 그것이 RAG입니다(실습 4).

In [ ]:
# 셀 6. 조립 — 규칙 4줄 + 찾은 조각 + 질문 = 프롬프트
RULES = '''[규칙]
1. 아래 규정집 조각만 근거로 답한다.
2. 조각에 없으면 "규정집에 없습니다"라고만 답한다.
3. 근거 조항 번호를 쓴다.
4. 3문장 이내로 답한다.'''

def build_prompt(question, k=3):
    found = search(question, k)
    body = "\n\n".join(f"({i}) {chunk}" for i, (_, _, chunk) in enumerate(found, 1))
    return f"{RULES}\n\n[규정집 조각]\n{body}\n\n[질문] {question}"

QUESTION = "국내 출장 숙박비 한도는 1박에 얼마인가요?"   # <-- 바꿔 보세요

print(build_prompt(QUESTION))

## 셀 7. 답하기 — 작은 생성 모델을 노트북 안에 내려받아 답하게 합니다 (GPU가 있을 때)
런타임 → 런타임 유형 변경 → **T4 GPU**가 잡혀 있으면 공개 모델 `Qwen/Qwen2.5-1.5B-Instruct`(약 3GB)를 내려받습니다(1~2분).
GPU가 없으면 이 셀은 안내문만 출력하고, 답은 셀 6의 프롬프트를 claude.ai에 붙여 받습니다. 셀 8 챗봇은 GPU가 없어도 동작합니다.

In [ ]:
# 셀 7. 답하기 — 생성 모델(작은 공개 LLM)을 붙입니다. GPU가 없으면 건너뜁니다.
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"           # 집에서 Mac으로 할 때
else:
    DEVICE = None

generator = None
if DEVICE:
    print(f"GPU 사용 ({DEVICE}) — 생성 모델을 내려받습니다 (1~2분)")
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(DEVICE)

    def generator(prompt):
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=200, do_sample=False)
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    print("생성 모델 준비 완료")
else:
    print("GPU 없음. 이 셀은 건너뛰고, 답은 셀 6의 프롬프트를 claude.ai에 붙여 받습니다.")

def answer(question, k=3):
    '''찾기 → 조립 → 답하기. 답 뒤에 근거 조각 제목을 붙인다.'''
    found = search(question, k)
    refs = ", ".join(title.split("(")[0] for _, title, _ in found)   # 제15조, 제16조, ...
    if generator is None:
        return "(생성 모델 없음) 아래 프롬프트를 claude.ai에 붙여 넣으세요.\n\n" + build_prompt(question, k)
    return generator(build_prompt(question, k)) + f"\n\n근거 조각: {refs}"

QUESTION = "국내 출장 숙박비 한도는 1박에 얼마인가요?"   # <-- 바꿔 보세요

print("질문:", QUESTION)
print("답:", answer(QUESTION))

## 셀 8. 챗봇 — 질문을 받아 답하기를 되풀이합니다
▶를 누르면 출력 맨 아래에 `질문>` 입력 칸이 생깁니다. 질문을 쓰고 Enter를 누르면 답(GPU가 없으면 붙여넣기용 프롬프트)과 **찾은 조각**이 나오고, 입력 칸이 다시 생깁니다.
끝내려면 빈 칸에서 Enter를 누르거나 셀 왼쪽의 ■를 누릅니다. 별도 화면이나 링크는 없습니다.

In [ ]:
# 셀 8. 챗봇 — 질문을 받아 답하기를 되풀이합니다 (GPU가 없어도 동작)
print("민트상사 규정집 챗봇입니다. 질문을 쓰고 Enter. 끝내려면 빈 칸에서 Enter.")

while True:
    print()
    question = input("질문> ").strip()
    if not question:
        print("챗봇을 마칩니다. 다시 하려면 이 셀을 ▶")
        break
    found = search(question, 3)
    if generator is None:
        print("답: (생성 모델이 없어 붙여넣기용 프롬프트를 보여 줍니다)\n")
        print(build_prompt(question))
    else:
        print("답:", generator(build_prompt(question)))
    print("[찾은 조각]")
    for rank, (score, title, _) in enumerate(found, 1):
        print(f"  {rank}위  점수 {score:.2f}  {title}")

## 셀 9 (선택). 내 문서로 바꾸기
`.txt` 또는 글자가 복사되는 `.pdf`를 올립니다(공개 문서·가상 문서만). 올린 뒤 **셀 3 → 셀 4**를 다시 실행하고 셀 8을 다시 ▶ 하면 챗봇이 새 문서로 답합니다.
조항 형식이 아닌 문서면 셀 3의 `CHUNK_MODE`를 `"글자수"`로 바꿉니다.

In [ ]:
# 셀 9 (선택). 내 문서로 바꾸기 — 파일을 올린 뒤 셀 3, 셀 4를 다시 실행합니다
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
if name.lower().endswith(".pdf"):
    from pypdf import PdfReader
    import io
    reader = PdfReader(io.BytesIO(uploaded[name]))
    DOCUMENT = "\n\n".join((page.extract_text() or "") for page in reader.pages)
else:
    DOCUMENT = uploaded[name].decode("utf-8", errors="ignore")

print(f"{name}: {len(DOCUMENT):,}자 읽음. 이제 셀 3과 셀 4를 다시 실행하세요.")
if not re.search(r"제\d+조\(", DOCUMENT):
    print('조항 형식이 아닙니다. 셀 3의 CHUNK_MODE = "글자수" 로 바꾸세요.')